# 🎮 Python Raycaster

A Wolfenstein-style 3D raycaster built entirely in Python using **NumPy** and **tkinter** — no game engine, no OpenGL.

This notebook walks through the engine section by section. Run the cells in order; the final cell launches the game window.

---

### Architecture overview

```
run_engine()
  └── loop()   (called every ~1 ms via tkinter.after)
        ├── apply_input()      read keys → move/rotate player
        ├── update_bob()       advance weapon head-bob
        ├── render()           DDA walls + floor/ceiling + lighting/fog
        ├── draw_sprites()     billboard NPCs with z-buffer
        └── draw_hud()         mini-map, debug text, weapon, crosshair
```

## 1. Imports

In [23]:
import numpy as np
import tkinter as tk
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageTk
import time

## 2. Custom math helpers

The engine implements its own `sin` and `cos` using Taylor series so the trig stays self-contained and readable.

$$\sin(x) = x - \frac{x^3}{3!} + \frac{x^5}{5!} - \cdots \qquad \cos(x) = 1 - \frac{x^2}{2!} + \frac{x^4}{4!} - \cdots$$

Both scalar (used for player rotation) and vectorised NumPy versions (used for arrays of ray angles) are provided.

In [24]:
PI     = 3.14159265358979
TWO_PI = 6.28318530717959

def norm(x):
    """Normalise an angle (radians) into the range [-π, π]."""
    x = x - TWO_PI * int(x / TWO_PI)
    if x < 0:
        x += TWO_PI
    if x > PI:
        x -= TWO_PI
    return x

def mc_sin(x):
    """Scalar sine via Taylor series (9 terms — accurate to ~15 significant figures)."""
    x = norm(x)
    t = x
    s = 0.0
    for n in range(1, 10):
        s += t
        t *= -(x * x) / ((2 * n) * (2 * n + 1))
    return s + t

def mc_cos(x):
    """Scalar cosine via Taylor series."""
    x = norm(x)
    t = 1.0
    s = 0.0
    for n in range(1, 10):
        s += t
        t *= -(x * x) / ((2 * n - 1) * (2 * n))
    return s + t

def scratch_floor(x):
    """Integer floor without importing math."""
    ix = int(x)
    return ix - 1 if x < ix else ix

def absolute(x):
    """Absolute value without importing math."""
    return x if x >= 0 else -x

def vec_sin(arr):
    """Element-wise sine for a NumPy array."""
    x = arr - TWO_PI * np.floor(arr / TWO_PI)
    X = np.where(x > PI, x - TWO_PI, x)
    t = X.copy()
    s = np.zeros_like(X)
    for n in range(1, 10):
        s += t
        t = t * (-(X * X) / ((2 * n) * (2 * n + 1)))
    return s + t

def vec_cos(arr):
    """Element-wise cosine for a NumPy array."""
    x = arr - TWO_PI * np.floor(arr / TWO_PI)
    x = np.where(x > PI, x - TWO_PI, x)
    t = np.ones_like(x)
    s = np.zeros_like(x)
    for n in range(1, 10):
        s += t
        t = t * (-(x * x) / ((2 * n - 1) * (2 * n)))
    return s + t

# Quick sanity check
import math
print(f'mc_sin(π/4) = {mc_sin(PI/4):.6f}  (expected {math.sin(math.pi/4):.6f})')
print(f'mc_cos(π/3) = {mc_cos(PI/3):.6f}  (expected {math.cos(math.pi/3):.6f})')

mc_sin(π/4) = 0.707107  (expected 0.707107)
mc_cos(π/3) = 0.500000  (expected 0.500000)


## 3. Constants & configuration

All engine parameters are collected here. Tweak these to change resolution, speed, lighting, or fog.

In [25]:
WIDTH  = 640
HEIGHT = 408
HALF_H = HEIGHT // 2

MOVE_SPEED = 0.08   # world units per frame
ROT_SPEED  = 0.05   # radians per frame

TEX_SIZE = 64
TEX_MASK = TEX_SIZE - 1

LIGHT_DECAY = 0.09
LIGHT_MIN   = 0.15

FOG_ONSET = 6.0
FOG_FULL  = 18.0
FOG_COLOR = np.array([30, 30, 35], dtype=np.float32)

BOB_FREQ = 2.8
BOB_AMP  = 12

## 4. World map & sprite list

The map is a 2D list of integers. `0` = open floor; `1–5` = wall types with different textures.  
Edit `RAW_MAP` to design your own levels — just keep the outer border fully walled.

In [26]:
RAW_MAP = [
    [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
    [1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1],
    [1,0,2,2,0,2,0,1,0,3,3,3,3,3,0,0,0,0,0,0,0,0,0,1],
    [1,0,2,0,0,2,0,4,0,3,0,0,0,3,0,0,0,0,0,0,0,0,0,1],
    [1,0,2,0,0,2,0,1,0,3,0,0,0,3,0,0,0,0,0,0,0,0,0,1],
    [1,0,2,2,0,2,0,1,0,3,3,0,3,3,0,0,0,0,0,0,0,0,0,1],
    [1,0,0,0,0,0,0,1,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,1],
    [1,1,1,4,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,1],
    [1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1],
    [1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,1],
    [1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1],
    [1,1,1,1,1,1,1,1,1,1,1,4,1,1,1,1,1,1,1,0,1,1,1,1],
    [1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1],
    [1,0,5,5,5,5,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1],
    [1,0,5,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1],
    [1,0,5,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1],
    [1,0,5,5,0,5,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1],
    [1,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1],
    [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
]

WORLD_MAP = np.array(RAW_MAP, dtype=np.int32)
MAP_ROWS, MAP_COLS = WORLD_MAP.shape

SPRITES = [
    ( 3.5, 11.5, 'person'),
    ( 4.5, 11.5, 'person'),
    ( 9.0,  4.5, 'person'),
    ( 9.0, 20.0, 'person'),
    (14.5, 10.5, 'person'),
]

print(f'Map: {MAP_ROWS} rows × {MAP_COLS} cols, {len(SPRITES)} sprites')

Map: 19 rows × 24 cols, 5 sprites


## 5. Asset directories & texture generators

Textures are generated procedurally on the first run and saved to disk. On subsequent runs the saved PNGs are loaded directly, so startup is instant.

In [27]:
for d in ['textures', 'floor', 'sprites', 'assets']:
    Path(d).mkdir(exist_ok=True)

def make_brick(rgb, s=TEX_SIZE):
    """Brick-wall texture with running-bond pattern and deterministic noise."""
    br, bg, bb = rgb
    y, x = np.mgrid[0:s, 0:s]
    row    = y // 10
    offset = np.where(row % 2 == 0, s // 4, 0)
    mortar = (y % 10 == 0) | ((x + offset) % (s // 4) == 0)
    noise  = ((x * 3 + y * 7) % 20) - 10
    tex    = np.zeros((s, s, 3), dtype=np.uint8)
    for ch, base in enumerate([br, bg, bb]):
        v = np.where(mortar, base - 40, base + noise)
        tex[:, :, ch] = np.clip(v, 0, 255)
    return tex

def make_checker(c1, c2, s=TEX_SIZE, n=8):
    """Checkerboard texture — used for floor and ceiling."""
    y, x = np.mgrid[0:s, 0:s]
    sq   = s // n
    tile = ((x // sq) + (y // sq)) % 2
    return np.where(tile[:, :, None] == 0, c1, c2).astype(np.uint8)

def make_person(s=TEX_SIZE):
    """Simple stick-figure person sprite (RGBA)."""
    t  = np.zeros((s, s, 4), dtype=np.uint8)
    cx = s // 2
    t[ 8:18, cx-5:cx+5] = [220, 180, 140, 255]
    t[18:40, cx-7:cx+7] = [180,  50,  50, 255]
    t[40:58, cx-7:cx-2] = [ 50,  50, 180, 255]
    t[40:58, cx+2:cx+7] = [ 50,  50, 180, 255]
    return t

def make_hand(h=150, w=200):
    """First-person weapon/hand sprite (RGBA)."""
    t = np.zeros((h, w, 4), dtype=np.uint8)
    t[70:h,  40:160] = [200, 160, 120, 255]
    t[40:80, 50:150] = [200, 160, 120, 255]
    t[20:55, 90:115] = [ 60,  60,  60, 255]
    t[30:50, 50: 65] = [200, 160, 120, 255]
    t[30:50,150:165] = [200, 160, 120, 255]
    return t

def load_rgb(path, make_fn):
    if Path(path).exists():
        return np.array(Image.open(path).convert('RGB').resize((TEX_SIZE, TEX_SIZE), Image.NEAREST), dtype=np.uint8)
    arr = make_fn()
    Image.fromarray(arr).save(path)
    print(f'  generated {path}')
    return arr

def load_rgba(path, make_fn, size=None):
    if Path(path).exists():
        img = Image.open(path).convert('RGBA')
        if size: img = img.resize(size, Image.NEAREST)
        return np.array(img, dtype=np.uint8)
    arr = make_fn()
    Image.fromarray(arr).save(path)
    print(f'  generated {path}')
    return arr

WALL_SPECS = {
    1: ('textures/wall_1.png', ( 90,  90,  90)),
    2: ('textures/wall_2.png', (150,  40,  40)),
    3: ('textures/wall_3.png', ( 40, 150,  40)),
    4: ('textures/wall_4.png', ( 40,  40, 150)),
    5: ('textures/wall_5.png', (150, 150,  40)),
}

WALL_TEX_STACK = np.zeros((6, TEX_SIZE, TEX_SIZE, 3), dtype=np.uint8)
for wid, (path, col) in WALL_SPECS.items():
    WALL_TEX_STACK[wid] = load_rgb(path, lambda c=col: make_brick(c))

FLOOR_TEX  = load_rgb('floor/floor.png',   lambda: make_checker((50,50,50),(35,35,35)))
CEIL_TEX   = load_rgb('floor/ceiling.png', lambda: make_checker((30,30,40),(20,20,30)))
PERSON_SPR = load_rgba('sprites/person.png', make_person, size=(TEX_SIZE, TEX_SIZE))
HAND_SPR   = load_rgba('assets/hand.png',   make_hand,   size=(200, 150))
SPR_TEX    = {'person': PERSON_SPR}

print('All textures loaded.')

All textures loaded.


## 6. Player class & rotation matrix

The player stores a **position** `(px, py)`, a **view direction** `(dx, dy)`, and a **camera plane** `(plx, ply)`.  
The camera plane is perpendicular to the direction and its length controls the horizontal FOV (~66° at length 0.66).

In [28]:
def rotation_2d(angle):
    """Return a 2×2 rotation matrix for the given angle (radians)."""
    c = mc_cos(angle)
    s = mc_sin(angle)
    return np.array([[c, -s],
                     [s,  c]], dtype=np.float64)

class Player:
    __slots__ = ('px', 'py', 'dx', 'dy', 'plx', 'ply', 'moving', 'bob_phase')

    def __init__(self, px=12.0, py=12.0, dx=-1.0, dy=0.0, plx=0.0, ply=0.66):
        self.px, self.py   = px, py
        self.dx, self.dy   = dx, dy
        self.plx, self.ply = plx, ply
        self.moving    = False
        self.bob_phase = 0.0

    def move(self, speed):
        nx = self.px + self.dx * speed
        ny = self.py + self.dy * speed
        if WORLD_MAP[int(nx), int(self.py)] == 0: self.px = nx
        if WORLD_MAP[int(self.px), int(ny)] == 0: self.py = ny
        self.moving = True

    def strafe(self, speed):
        nx = self.px - self.dy * speed
        ny = self.py + self.dx * speed
        if WORLD_MAP[int(nx), int(self.py)] == 0: self.px = nx
        if WORLD_MAP[int(self.px), int(ny)] == 0: self.py = ny
        self.moving = True

    def rotate(self, angle):
        R = rotation_2d(angle)
        d = R @ np.array([self.dx,  self.dy ])
        p = R @ np.array([self.plx, self.ply])
        self.dx,  self.dy  = float(d[0]), float(d[1])
        self.plx, self.ply = float(p[0]), float(p[1])

## 7. Renderer

### How DDA works

For each screen column a ray is cast from the player's position. The algorithm tracks how far the ray must travel to cross the next **X** grid line and the next **Y** grid line, then steps across whichever crossing is closer — repeating until a wall cell is hit.

The **perpendicular** distance (not the Euclidean distance to the hit point) is used to calculate the projected wall height, which eliminates fish-eye distortion.

All 640 rays are processed simultaneously using NumPy broadcasting.

In [29]:
_COL       = np.arange(WIDTH, dtype=np.float64)
_CAMX      = 2.0 * _COL / WIDTH - 1.0
_FLOOR_ROW = np.arange(HALF_H + 1, HEIGHT, dtype=np.float64)

def render(player, frame):
    """Fill frame with walls, floor, ceiling, lighting and fog. Returns z-buffer."""
    px, py   = player.px, player.py
    dx, dy   = player.dx, player.dy
    plx, ply = player.plx, player.ply

    # ── Floor & ceiling ──────────────────────────────────────────────────────
    row_dist = HALF_H / (_FLOOR_ROW - HALF_H)
    step_x   = row_dist * (2.0 * plx / WIDTH)
    step_y   = row_dist * (2.0 * ply / WIDTH)
    fx0 = px + row_dist * (dx - plx)
    fy0 = py + row_dist * (dy - ply)
    col_range = np.arange(WIDTH, dtype=np.float64)
    fx = fx0[:, None] + step_x[:, None] * col_range[None, :]
    fy = fy0[:, None] + step_y[:, None] * col_range[None, :]
    tx = np.clip((fx * TEX_SIZE).astype(np.int32) & TEX_MASK, 0, TEX_MASK)
    ty = np.clip((fy * TEX_SIZE).astype(np.int32) & TEX_MASK, 0, TEX_MASK)
    frame[HALF_H + 1:HEIGHT, :] = FLOOR_TEX[ty, tx]
    ceil_y = (HEIGHT - 1 - _FLOOR_ROW).astype(int)
    frame[ceil_y, :] = CEIL_TEX[ty, tx]

    # ── Wall DDA ─────────────────────────────────────────────────────────────
    rdx = dx + plx * _CAMX
    rdy = dy + ply * _CAMX
    mx  = np.full(WIDTH, int(px), dtype=np.int32)
    my  = np.full(WIDTH, int(py), dtype=np.int32)
    with np.errstate(divide='ignore', invalid='ignore'):
        dlx = np.where(rdx == 0, 1e30, np.abs(1.0 / rdx))
        dly = np.where(rdy == 0, 1e30, np.abs(1.0 / rdy))
    stx = np.where(rdx < 0, -1, 1).astype(np.int32)
    sty = np.where(rdy < 0, -1, 1).astype(np.int32)
    sdx = np.where(rdx < 0, (px - mx) * dlx, (mx + 1.0 - px) * dlx)
    sdy = np.where(rdy < 0, (py - my) * dly, (my + 1.0 - py) * dly)
    hit  = np.zeros(WIDTH, dtype=bool)
    side = np.zeros(WIDTH, dtype=np.int32)
    for _ in range(MAP_ROWS + MAP_COLS):
        if hit.all(): break
        alive = ~hit
        go_x  = alive & (sdx < sdy)
        go_y  = alive & ~go_x
        sdx   = np.where(go_x, sdx + dlx, sdx)
        mx    = np.where(go_x, mx + stx, mx)
        side  = np.where(go_x, 0, side)
        sdy   = np.where(go_y, sdy + dly, sdy)
        my    = np.where(go_y, my + sty, my)
        side  = np.where(go_y, 1, side)
        cmx   = np.clip(mx, 0, MAP_ROWS - 1)
        cmy   = np.clip(my, 0, MAP_COLS - 1)
        hit  |= (WORLD_MAP[cmx, cmy] > 0)

    # ── Project walls ─────────────────────────────────────────────────────────
    perp       = np.maximum(np.where(side == 0, sdx - dlx, sdy - dly), 0.0001)
    z_buffer   = perp.copy()
    wall_h     = np.maximum(1, (HEIGHT / perp).astype(np.int32))
    draw_start = np.maximum(0,      -wall_h // 2 + HALF_H)
    draw_end   = np.minimum(HEIGHT,  wall_h // 2 + HALF_H)
    wall_hit   = np.where(side == 0, py + perp * rdy, px + perp * rdx)
    wall_hit  -= np.floor(wall_hit)
    tex_x      = (wall_hit * TEX_SIZE).astype(np.int32)
    flip       = ((side == 0) & (rdx > 0)) | ((side == 1) & (rdy < 0))
    tex_x      = np.clip(np.where(flip, TEX_MASK - tex_x, tex_x), 0, TEX_MASK)
    cmx        = np.clip(mx, 0, MAP_ROWS - 1)
    cmy        = np.clip(my, 0, MAP_COLS - 1)
    wall_ids   = np.clip(WORLD_MAP[cmx, cmy], 1, 5)
    brightness = np.clip(np.exp(-LIGHT_DECAY * perp), LIGHT_MIN, 1.0)
    brightness = np.where(side == 1, brightness * 0.65, brightness)
    fog        = np.clip((perp - FOG_ONSET) / (FOG_FULL - FOG_ONSET), 0.0, 1.0)

    # ── Rasterise columns ─────────────────────────────────────────────────────
    for x in range(WIDTH):
        ds, de = int(draw_start[x]), int(draw_end[x])
        if de <= ds: continue
        n      = de - ds
        wh     = max(1, int(wall_h[x]))
        tx     = int(tex_x[x])
        tex    = WALL_TEX_STACK[int(wall_ids[x])]
        ty_f   = (np.arange(n, dtype=np.float32) + (ds - HALF_H + wh * 0.5)) * TEX_SIZE / wh
        ty     = np.clip(ty_f.astype(np.int32), 0, TEX_MASK)
        colour = tex[ty, tx].astype(np.float32) * brightness[x]
        f = fog[x]
        if f > 0.0:
            colour = colour * (1.0 - f) + FOG_COLOR * f
        frame[ds:de, x] = np.clip(colour, 0, 255).astype(np.uint8)

    return z_buffer

## 8. Sprite renderer

Sprites are **billboards**: they always face the camera.  
They are sorted back-to-front (painter's algorithm) and drawn column by column; any column where a wall is closer than the sprite (checked against the z-buffer) is skipped.

In [30]:
def draw_sprites(player, frame, z_buffer):
    px, py   = player.px, player.py
    dx, dy   = player.dx, player.dy
    plx, ply = player.plx, player.ply
    inv_det  = 1.0 / (plx * dy - dx * ply + 1e-30)
    ordered  = sorted(SPRITES, key=lambda s: (s[0]-px)**2 + (s[1]-py)**2, reverse=True)

    for (sx, sy, stype) in ordered:
        tex = SPR_TEX.get(stype)
        if tex is None: continue
        th, tw = tex.shape[:2]
        rx, ry  = sx - px, sy - py
        cam_xs  =  inv_det * ( dy * rx - dx * ry)
        cam_z   =  inv_det * (-ply * rx + plx * ry)
        if cam_z <= 0.1: continue
        scr_x  = int((WIDTH / 2) * (1 + cam_xs / cam_z))
        proj_h = abs(int(HEIGHT / cam_z))
        proj_w = proj_h
        if proj_h == 0: continue
        ds_y = max(0,      -proj_h // 2 + HALF_H)
        de_y = min(HEIGHT,  proj_h // 2 + HALF_H)
        ds_x = max(0,       scr_x - proj_w // 2)
        de_x = min(WIDTH,   scr_x + proj_w // 2)
        if de_y <= ds_y or de_x <= ds_x: continue
        screen_ys = np.arange(ds_y, de_y, dtype=np.float32)
        ty = np.clip(((screen_ys - (HALF_H - proj_h//2)) * th / proj_h).astype(np.int32), 0, th-1)
        for x in range(ds_x, de_x):
            if cam_z >= z_buffer[x]: continue
            tx      = max(0, min(tw-1, int((x - (scr_x - proj_w//2)) * tw / proj_w)))
            alpha   = tex[ty, tx, 3]
            visible = alpha >= 128
            if not visible.any(): continue
            rows = screen_ys[visible].astype(np.int32)
            frame[rows, x] = tex[ty[visible], tx, :3]

## 9. HUD

The HUD draws directly into the frame array each tick:
- **Mini-map** — top-right corner, walls colour-coded, player dot + direction arrow
- **Debug panel** — position, direction, wall distance, FPS
- **Weapon hand** — right-aligned with a `sin`-driven bob when walking
- **Crosshair** — centred on screen

In [31]:
try:
    _FONT = ImageFont.truetype('arial.ttf', 14)
except Exception:
    _FONT = ImageFont.load_default()

_MM_CELL = 5
_MM_PAD  = 10
_MM_W    = MAP_COLS * _MM_CELL
_MM_H    = MAP_ROWS * _MM_CELL
_MM_OX   = WIDTH  - _MM_W - _MM_PAD
_MM_OY   = _MM_PAD

_WCOLORS = {1:(100,100,100), 2:(150,50,50), 3:(50,150,50), 4:(50,50,150), 5:(150,150,50)}

_mm_base = np.zeros((_MM_H, _MM_W, 3), dtype=np.uint8)
for _r in range(MAP_ROWS):
    for _c in range(MAP_COLS):
        _v = WORLD_MAP[_r, _c]
        if _v > 0:
            _y0, _x0 = _r*_MM_CELL, _c*_MM_CELL
            _mm_base[_y0:_y0+_MM_CELL, _x0:_x0+_MM_CELL] = _WCOLORS.get(_v, (80,80,80))
_mm_mask = (_mm_base > 0).any(axis=2, keepdims=True)

def _fwd_dist(player):
    px, py   = player.px, player.py
    rdx, rdy = player.dx, player.dy
    mx, my   = int(px), int(py)
    dlx = 1e30 if rdx == 0 else absolute(1.0 / rdx)
    dly = 1e30 if rdy == 0 else absolute(1.0 / rdy)
    stx = -1 if rdx < 0 else 1
    sty = -1 if rdy < 0 else 1
    sdx = (px - mx)*dlx if rdx < 0 else (mx + 1.0 - px)*dlx
    sdy = (py - my)*dly if rdy < 0 else (my + 1.0 - py)*dly
    side = 0
    for _ in range(MAP_ROWS + MAP_COLS):
        if sdx < sdy: sdx += dlx; mx += stx; side = 0
        else:         sdy += dly; my += sty; side = 1
        if 0 <= mx < MAP_ROWS and 0 <= my < MAP_COLS and WORLD_MAP[mx, my] > 0: break
    return (sdx - dlx) if side == 0 else (sdy - dly)

def update_bob(player, dt):
    if player.moving:
        player.bob_phase += TWO_PI * BOB_FREQ * dt
        if player.bob_phase > TWO_PI: player.bob_phase -= TWO_PI
    else:
        player.bob_phase *= max(0.0, 1.0 - dt * BOB_FREQ * 3.0)
    player.moving = False

def draw_hud(player, frame, z_buffer, fps):
    oy, ox = _MM_OY, _MM_OX
    region = frame[oy:oy+_MM_H, ox:ox+_MM_W]
    bg = (region * 0.25).astype(np.uint8)
    frame[oy:oy+_MM_H, ox:ox+_MM_W] = np.where(_mm_mask, _mm_base, bg)
    pdx = np.clip(int(ox + player.py * _MM_CELL), ox, ox+_MM_W-1)
    pdy = np.clip(int(oy + player.px * _MM_CELL), oy, oy+_MM_H-1)
    frame[pdy-2:pdy+3, pdx-2:pdx+3] = (255, 50, 50)
    ax  = int(pdx + player.dy * 9)
    ay  = int(pdy + player.dx * 9)
    ts  = np.linspace(0, 1, 10)
    lxs = np.clip((pdx + ts*(ax-pdx)).astype(int), ox, ox+_MM_W-1)
    lys = np.clip((pdy + ts*(ay-pdy)).astype(int), oy, oy+_MM_H-1)
    frame[lys, lxs] = (60, 230, 230)
    fwd    = _fwd_dist(player)
    TW, TH = 270, 88
    surf = Image.new('RGB', (TW, TH), (0,0,0))
    d = ImageDraw.Draw(surf)
    d.text(( 8,  4), f'POS  X:{player.px:.2f}  Y:{player.py:.2f}', fill=(220,220,220), font=_FONT)
    d.text(( 8, 24), f'DIR  X:{player.dx:.2f}  Y:{player.dy:.2f}', fill=(220,220,220), font=_FONT)
    d.text(( 8, 44), f'WALL DIST: {fwd:.2f} units',                fill=(255,200, 50), font=_FONT)
    d.text(( 8, 64), f'FPS: {fps}',                                fill=(100,220,100), font=_FONT)
    txt = np.array(surf, dtype=np.uint8)
    frame[6:6+TH, 6:6+TW] = (frame[6:6+TH, 6:6+TW] * 0.30 + txt * 0.70).astype(np.uint8)
    hh, hw = HAND_SPR.shape[:2]
    hx = WIDTH - hw - 8
    bob_offset = int(BOB_AMP * mc_sin(player.bob_phase))
    hy = max(0, min(HEIGHT - hh, HEIGHT - hh - 4 + bob_offset))
    if hx >= 0:
        a   = HAND_SPR[:,:,3:4].astype(np.float32) / 255.0
        rgb = HAND_SPR[:,:,:3].astype(np.float32)
        roi = frame[hy:hy+hh, hx:hx+hw].astype(np.float32)
        frame[hy:hy+hh, hx:hx+hw] = (rgb * a + roi * (1.0 - a)).astype(np.uint8)
    cx, cy = WIDTH // 2, HALF_H
    frame[cy,      cx-8:cx+9] = (200, 200, 200)
    frame[cy-8:cy+9,  cx   ] = (200, 200, 200)

## 10. Input & game loop

Key state is tracked in a `set`; `apply_input` reads it once per frame and moves the player accordingly.

In [32]:
KEYS_HELD: set = set()

def _on_key_press(event):   KEYS_HELD.add(event.keysym.lower())
def _on_key_release(event): KEYS_HELD.discard(event.keysym.lower())

def apply_input(player):
    if 'escape' in KEYS_HELD or 'q' in KEYS_HELD: return False
    if 'w'     in KEYS_HELD or 'up'    in KEYS_HELD: player.move( MOVE_SPEED)
    if 's'     in KEYS_HELD or 'down'  in KEYS_HELD: player.move(-MOVE_SPEED)
    if 'a'     in KEYS_HELD:                          player.strafe(MOVE_SPEED)
    if 'd'     in KEYS_HELD:                          player.strafe(-MOVE_SPEED)
    if 'left'  in KEYS_HELD:                          player.rotate( ROT_SPEED)
    if 'right' in KEYS_HELD:                          player.rotate(-ROT_SPEED)
    return True

## 11. Run the engine

▶ **Run this cell to launch the game window.**

The loop is driven by `tkinter.after(1, loop)` so the GUI event queue stays responsive between frames. Press **Q** or **Esc** to quit cleanly.

In [33]:
def run_engine():
    player    = Player()
    frame     = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
    root      = tk.Tk()
    root.title('DOOM Raycaster  |  WASD / Arrows  |  Q = quit')
    root.resizable(False, False)
    canvas    = tk.Canvas(root, width=WIDTH, height=HEIGHT, bg='black', highlightthickness=0)
    canvas.pack()
    root.bind('<KeyPress>',   _on_key_press)
    root.bind('<KeyRelease>', _on_key_release)
    photo_ref = [None]
    fps_val   = [0];  fps_cnt = [0]
    fps_t     = [time.perf_counter()]
    last_t    = [time.perf_counter()]

    def loop():
        now       = time.perf_counter()
        dt        = now - last_t[0]
        last_t[0] = now
        if not apply_input(player):
            root.destroy(); return
        update_bob(player, dt)
        frame[:] = 0
        z = render(player, frame)
        draw_sprites(player, frame, z)
        draw_hud(player, frame, z, fps_val[0])
        img = ImageTk.PhotoImage(Image.fromarray(frame, 'RGB'))
        photo_ref[0] = img
        canvas.create_image(0, 0, anchor=tk.NW, image=img)
        fps_cnt[0] += 1
        elapsed = now - fps_t[0]
        if elapsed >= 0.5:
            fps_val[0] = int(fps_cnt[0] / elapsed)
            fps_cnt[0] = 0
            fps_t[0]   = now
        if canvas.winfo_exists():
            canvas.after(1, loop)

    canvas.after(0, loop)
    print('Click the window to focus, then use WASD / arrow keys.')
    root.mainloop()
    print('Engine stopped.')

run_engine()

Click the window to focus, then use WASD / arrow keys.
Engine stopped.
